# 02 — Feature Engineering

**Tujuan:** Mengubah data tabular job (yang sudah dibersihkan di notebook 01) menjadi representasi numerik yang bisa dihitung kesamaannya — yaitu **binary skill matrix** dan **TF-IDF matrix** dari deskripsi.

**Output:**
- Skill vocabulary (list of unique skills)
- Job-skill binary matrix (n_jobs × n_skills)
- TF-IDF matrix (n_jobs × n_features) dari description

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from src.feature_engineering import (
    parse_skills_list, normalize_skill_name,
    build_skill_vocabulary, build_job_skill_matrix,
    build_user_vector, build_tfidf_matrix
)
from src.data_loader import load_jobs_data

sns.set_style('whitegrid')
LI_BLUE = '#0A66C2'

## 2.1 Load Cleaned Data

In [ ]:
df, is_demo = load_jobs_data()
print(f'Loaded {len(df):,} job postings ({"demo synthetic" if is_demo else "real LinkedIn data"})')
df[['title', 'skills_list']].head(3)

## 2.2 Build Skill Vocabulary

Vocabulary = list unik semua skill yang muncul minimal `min_freq` kali di dataset.
Skill yang terlalu jarang dianggap noise dan tidak masuk vocabulary.

In [ ]:
vocab = build_skill_vocabulary(df, skill_col='skills_list', min_freq=5)
print(f'Vocabulary size: {len(vocab)} unique skills')
print(f'Sample vocabulary: {vocab[:20]}')

## 2.3 Skill Frequency Distribution

In [ ]:
all_skills = []
for skills in df['skills_list']:
    parsed = parse_skills_list(skills)
    all_skills.extend([normalize_skill_name(s) for s in parsed if s])

skill_counts = Counter(all_skills)
top_20 = skill_counts.most_common(20)

df_top = pd.DataFrame(top_20, columns=['skill', 'count'])
df_top['skill'] = df_top['skill'].str.title()

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(df_top['skill'][::-1], df_top['count'][::-1], color=LI_BLUE)
ax.set_xlabel('Job posting count')
ax.set_title('Top 20 Most Demanded Skills in IT')
plt.tight_layout()
plt.show()

## 2.4 Build Binary Job-Skill Matrix

Setiap baris = job, setiap kolom = skill di vocabulary. Cell bernilai 1 jika job memerlukan skill itu, 0 jika tidak.

In [ ]:
job_matrix = build_job_skill_matrix(df, vocab, skill_col='skills_list')
print(f'Job-skill matrix shape: {job_matrix.shape}')
print(f'Density (% non-zero): {(job_matrix.sum() / job_matrix.size * 100):.2f}%')
print(f'Avg skills per job: {job_matrix.sum(axis=1).mean():.1f}')
print(f'Max skills per job: {job_matrix.sum(axis=1).max()}')

### Visualize matrix sparsity

In [ ]:
# Sample 50 jobs and 50 skills untuk visualisasi
import numpy as np
np.random.seed(42)
sample_jobs = np.random.choice(len(df), 50, replace=False)
sample_skills = np.random.choice(len(vocab), 50, replace=False)

submatrix = job_matrix[sample_jobs][:, sample_skills]

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(submatrix, cmap='Blues', aspect='auto')
ax.set_xlabel('Skill index (sampled)')
ax.set_ylabel('Job index (sampled)')
ax.set_title('Sample of Binary Job-Skill Matrix (50x50)')
plt.tight_layout()
plt.show()

## 2.5 Build User Profile Vector

Skill yang dipilih user diubah menjadi binary vector dengan dimensi yang sama dengan job matrix, supaya bisa dihitung cosine similarity.

In [ ]:
sample_user_skills = ['Python', 'SQL', 'Machine Learning', 'Statistics', 'Pandas']
user_vec = build_user_vector(sample_user_skills, vocab)
print(f'User vector shape: {user_vec.shape}')
print(f'Skills active in user vector: {user_vec.sum()} / {len(vocab)}')

# Show which skills landed in vocab
active_idx = np.where(user_vec.flatten() == 1)[0]
print('\nMatched skills:')
for i in active_idx:
    print(f'  [{i}] {vocab[i]}')

## 2.6 TF-IDF on Job Descriptions

Sebagai fitur tambahan, kita ekstrak TF-IDF dari kolom description untuk menangkap konteks selain skill list (misalnya: "junior", "remote", "healthcare").

In [ ]:
tfidf_matrix, vectorizer = build_tfidf_matrix(df, text_col='description', max_features=300)
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'Sample features (top 30): {vectorizer.get_feature_names_out()[:30].tolist()}')

## Kesimpulan Feature Engineering

- Skill vocabulary terdiri dari skill-skill yang muncul cukup sering — noise dari skill yang sangat jarang sudah disaring
- Job-skill matrix berukuran besar tapi sangat sparse (rata-rata <10% kolom aktif per baris) — wajar untuk binary feature
- User vector dibangun dengan dimensi yang sama, siap untuk dot product / cosine similarity
- TF-IDF menangkap konteks tambahan di luar skill list eksplisit